In [11]:
import sqlite3
import requests

In [ ]:
conn = sqlite3.connect("data.db") # edit with correct filepath
cur = conn.cursor()

In [3]:
errors = []

def update_book(bookid):
  update_query = "UPDATE books SET coverid = ? WHERE bookId = ?"
  select_query = "SELECT title, authors FROM books WHERE bookId = ?"

  cur.execute(select_query, (bookid,))
  res = cur.fetchone()
  
  title = res[0]
  authors = res[1]
  authors = authors.replace(',', '')

  # clean title by removing brackets and text within
  while '(' in title or ')' in title:
    open_bracket = title.find('(')
    close_bracket = title.find(')')
    title = title[:open_bracket] + title[close_bracket+1:]

  keywords = title.split(' ')
  # append authors names
  keywords += authors.split(' ')

  q = "+".join(keywords)

  url = f"https://openlibrary.org/search.json?q={q}"
  headers = {
      "User-Agent": "DBExtension/1.0 (teamtheminecraftshow@gmail.com)"
  }

  try: 
    response = requests.get(url, headers=headers)
    docs = response.json()['docs']
    found = False
    
    # assume first full reseult is the book we want
    for doc in docs:
      if 'cover_i' in doc:
        cover_i = doc['cover_i']
        found = True
        break
    
    # check if got coverid at all
    if found:
      cur.execute(update_query, (cover_i, bookid,))
      conn.commit()
    else:
      errors.append(bookid)
  except:
    errors.append(bookid)

  

In [ ]:
errors = []
num_books = 10000

# data.db bookid is 1 indexed
for i in range(1, num_books+1):
  update_book(i)
  print(f"Current Book: {i}; Current no of errors: {len(errors)}/{num_books}", end="\r")

In [10]:
conn.close()